# Preprocessing Data for NLP Tasks

## Setup

In [1]:
!python --version

Python 3.12.4


In [2]:
!cat requirements.txt

# Python 3.12.x
pandas==2.3.3
numpy==1.26.4
matplotlib==3.10.6
seaborn==0.13.2
wordcloud==1.9.4
nltk==3.9.2
spacy==3.8.9
textblob==0.19.0
scikit-learn==1.7.2
bertopic==0.17.3
pyLDAvis==3.4.1
gensim==4.4.0


In [3]:
!python -m pip install -q -r requirements.txt

In [4]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.7 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### Imports

In [5]:
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from bertopic import BERTopic
import pandas as pd
from nlp import (
    group_topics_lda,
    get_topic_lda,
    visualize_topics_lda,
    remove_stopwords,
    preprocess_data
)

2025-11-30 12:14:18.351525: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-30 12:14:18.382818: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764522858.403335  148857 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764522858.408677  148857 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-30 12:14:18.443145: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

[nltk_data] Downloading package stopwords to /home/ballen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Configurations

In [6]:
data_path = '../data/cleaned_ntsb.csv'

In [7]:
pd.set_option('display.max_columns', None)

## Load Data

In [8]:
df = pd.read_csv(data_path, low_memory=False)
df.shape

(87951, 47)

In [9]:
df.head()

,event_id,event_date,investigation_type,country,aircraft_damage,aircraft_category,make,model,amateur_built,number_of_engines,engine_type,far_description,schedule,purpose_of_flight,total_fatal_injuries,total_serious_injuries,total_minor_injuries,total_uninjured,weather_condition,broad_phase_of_flight,analysis,analysis_clean,city,longitude,latitude,address,geometry,place,number_of_seats,type_aircraft,type_engine,total_person,far_description_factorized,schedule_factorized,purpose_of_flight_factorized,make_factorized,model_factorized,year,publication_year,month,publication_month,day,publication_day,date_difference,publication_month_name,event_month_name,season
0,20001218X45444,1948-10-24,Accident,United States,Destroyed,fixed wing single engine,stinson,108-3,No,1,reciprocating,part 91: general aviation,UNK,Personal,2,0,0,0,UNK,Cruise,"ON OCTOBER 24, 1948, THE PILOT DEPARTED A PRIV...",on october the pilot departed a private airstr...,moose creek,-147.160665,64.713512,"Moose Creek, Fairbanks North Star, Alaska, Uni...",POINT (-147.1606646266588 64.7135123),mountain,4,4,1,2,1,2,0,0,0,1948,2001,10,8,24,24.0,26,August,October,Fall
1,20001218X45447,1962-07-19,Accident,United States,Destroyed,weight-shift-control,piper,pa24-180,No,1,reciprocating,part 91: general aviation,UNK,Personal,4,0,0,0,UNK,Unknown,"ON JULY 19, 1962, A COMMERCIAL, NON-INSTRUMENT...",on july a commercial non instrument rated pilo...,bridgeport,-73.188786,41.179269,"Bridgeport, Greater Bridgeport Planning Region...",POINT (-73.1887863 41.1792695),sea,4,7,1,4,1,2,0,1,1,1962,1996,7,9,19,19.0,34,September,July,Summer
2,20061025X01555,1974-08-30,Accident,United States,Destroyed,fixed wing single engine,cessna,172m,No,1,reciprocating,part 91: general aviation,UNK,Personal,3,0,0,1,IMC,Cruise,The private pilot was issued his certificate o...,the private pilot was issued his certificate o...,saltville,-81.762063,36.881503,"Saltville, Smyth County, Virginia, United States",POINT (-81.7620635 36.8815031),sea,4,4,1,3,1,2,0,2,2,1974,2007,8,2,30,30.0,33,February,August,Summer
3,20001218X45448,1977-06-19,Accident,United States,Destroyed,weight-shift-control,rockwell,112,No,1,reciprocating,part 91: general aviation,UNK,Personal,2,0,0,0,IMC,Cruise,The aircraft wreckage was discovered 22 miles ...,the aircraft wreckage was discovered miles sou...,eureka,-124.167375,40.790687,"Eureka, Humboldt County, California, United St...",POINT (-124.1673746 40.7906871),airport,4,7,1,2,1,2,0,3,3,1977,2000,6,12,19,19.0,23,December,June,Summer
4,20041105X01764,1979-08-02,Accident,United States,Destroyed,fixed wing multi engine,cessna,501,No,2,turbo fan,part 91: general aviation,UNK,Personal,1,2,0,0,VMC,Approach,The Safety Board's full report is available at...,the safety board s full report is available at...,canton,-95.864051,32.555664,"Canton, Van Zandt County, Texas, 75103, United...",POINT (-95.8640507 32.555664),airport,8,5,5,3,1,2,0,2,4,1979,1980,8,4,2,2.0,1,April,August,Summer


## Preprocess Narratives
Remove stopwords and special tokens for anlaysis and data visualization purposes.

In [10]:
tokens = remove_stopwords([df['analysis_clean'].iloc[0]])
sent = ' '.join(tokens[0])
print(sent)

october pilot departed private airstrip moose creek ranch two pilots departed time reported encountering heavy snow squalls along selway river headed westbound wreckage found april miles west moose creek selway river drainage mountainous terrain wreckage exhibited characteristics consistent impact trees terrain cruise flight


In [11]:
results = remove_stopwords(df['analysis_clean'].tolist())
len(results)

87951

In [12]:
results = [ ' '.join(result) for result in results ]

In [13]:
print(results[0])

october pilot departed private airstrip moose creek ranch two pilots departed time reported encountering heavy snow squalls along selway river headed westbound wreckage found april miles west moose creek selway river drainage mountainous terrain wreckage exhibited characteristics consistent impact trees terrain cruise flight


In [14]:
df.insert(df.columns.get_loc("analysis_clean")+1, "analysis_preprocessed", results)

In [15]:
df[['analysis_clean', 'analysis_preprocessed']].head(3).values

array([['on october the pilot departed a private airstrip at the moose creek ranch at two other pilots that departed at the same time reported encountering heavy snow squalls along the selway river as they headed westbound the wreckage was found in april miles west of moose creek in the selway river drainage in mountainous terrain the wreckage exhibited characteristics consistent with impact with trees and terrain in cruise flight ',
        'october pilot departed private airstrip moose creek ranch two pilots departed time reported encountering heavy snow squalls along selway river headed westbound wreckage found april miles west moose creek selway river drainage mountainous terrain wreckage exhibited characteristics consistent impact trees terrain cruise flight'],
       ['on july a commercial non instrument rated pilot and three passengers were crossing high mountainous terrain on a night cross country flight subsequently the airplane collided with rising terrain at about feet mean 

## Sentiment Analysis

In [16]:
df["sentiment_polarity"] = df["analysis_clean"].apply(lambda x: TextBlob(x).sentiment.polarity)
df["sentiment_category"] = pd.cut(
    df["sentiment_polarity"],
    bins=[-1, -0.05, 0.05, 1],
    labels=["Negative", "Neutral", "Positive"]
)

In [17]:
df[["analysis_clean", "sentiment_polarity", "sentiment_category"]].sample(50, random_state=42)

,analysis_clean,sentiment_polarity,sentiment_category
25609,witnesses observed the acft maneuvering at a l...,-0.019136,Neutral
31738,factual,0.000000,Neutral
5346,after takeoff the plt of the ultralight vehicl...,0.020833,Neutral
85707,the pilot had made two flights on the day of t...,0.024351,Neutral
59785,the airplane was flown by the student and cert...,0.024451,Neutral
22491,the plt landed in a field which he knew contai...,0.214286,Positive
25703,the pilot was landing and bounced twice he rej...,0.158929,Positive
7492,the plt angled in on the final approach at low...,0.095238,Positive
64569,same as factual information,0.000000,Neutral
4293,the pilot was warming the eng for a compressio...,0.037143,Neutral


### Analyze Sentiment Spread

In [18]:
sentiment_counts = df["sentiment_category"].value_counts()
sentiment_ratio = sentiment_counts / len(df)
sentiment_percent = sentiment_ratio.map(lambda x: f'{x*100:.2f}%')
sentiment_spread = pd.DataFrame({
    'Count': sentiment_counts,
    'Ratio': sentiment_ratio,
    'Percentage': sentiment_percent,
})
sentiment_spread

,Count,Ratio,Percentage
sentiment_category,,,
Neutral,38948,0.442837,44.28%
Positive,27845,0.316597,31.66%
Negative,21158,0.240566,24.06%


**Observation:** Slight imbalance of sentiment categories across the dataset with `Neutral` sentiment representing the majority and `Negative` representing the minority. A perfect balance would have been approximately `33.33%` per category.

In [19]:
# Inspect one report from each category
for category in df["sentiment_category"].unique():
    item = df[df["sentiment_category"] == category].iloc[0]
    narrative = item['analysis_clean']
    event_id = item['event_id']
    event_date = item['event_date']
    print('='*60)
    print(f'Category   : {category}')
    print(f'Event ID   : {event_id}')
    print(f'Event Date : {event_date}')
    print(f'Narrative  : {narrative}')
print('='*60)

Category   : Neutral
Event ID   : 20001218X45444
Event Date : 1948-10-24
Narrative  : on october the pilot departed a private airstrip at the moose creek ranch at two other pilots that departed at the same time reported encountering heavy snow squalls along the selway river as they headed westbound the wreckage was found in april miles west of moose creek in the selway river drainage in mountainous terrain the wreckage exhibited characteristics consistent with impact with trees and terrain in cruise flight 
Category   : Negative
Event ID   : 20001218X45447
Event Date : 1962-07-19
Narrative  : on july a commercial non instrument rated pilot and three passengers were crossing high mountainous terrain on a night cross country flight subsequently the airplane collided with rising terrain at about feet mean sea level wreckage was found on a degree slope with extensive damage the airplane was reported missing and went undiscovered until august the airplane with human remains was discovered i

## Keyword Extraction

In [20]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=10, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["analysis_clean"])
feature_names = vectorizer.get_feature_names_out()

In [21]:
# Get top words by average TF-IDF score
tfidf_mean = tfidf_matrix.mean(axis=0).A1
keywords = pd.DataFrame({"word": feature_names, "tfidf": tfidf_mean})
keywords.sort_values("tfidf", ascending=False).head(20)

,word,tfidf
333,airplane,0.068049
6847,pilot,0.059682
5243,landing,0.039301
3253,engine,0.038957
8128,runway,0.037438
3993,fuel,0.035504
3552,factual,0.032002
5329,left,0.030863
97,acft,0.030309
3804,flight,0.029862


## Topic Modeling

### BERT Topic

In [22]:
topic_model = BERTopic(language="english")
topics, probs = topic_model.fit_transform(df["analysis_clean"])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/home/ballen/anaconda3/envs/data-science/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning: This process (pid=148857) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()
/home/ballen/anaconda3/envs/data-science/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: DeprecationWarning: This process (pid=148857) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you c

In [23]:
df["topic"] = topics
topic_info = topic_model.get_topic_info()
topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25639,-1_airplane_runway_pilot_and,"[airplane, runway, pilot, and, the, engine, to...",[the private pilot was conducting a personal c...
1,0,14350,0_acft_plt_rwy_flt,"[acft, plt, rwy, flt, eng, arpt, ft, aprx, apc...",[while on final apch to land at a private arpt...
2,1,6368,1_helicopter_rotor_tail_collective,"[helicopter, rotor, tail, collective, autorota...",[the helicopter pilot was lifting off with an ...
3,2,3774,2_student_instructor_solo_cfi,"[student, instructor, solo, cfi, runway, go, a...",[the flight instructor and student pilot were ...
4,3,3414,3_factual_same_information_as,"[factual, same, information, as, narrative, fo...","[same as factual information, same as factual ..."
5,4,2681,4_fuel_tank_gallons_tanks,"[fuel, tank, gallons, tanks, selector, engine,...",[the private pilot reported that when he depar...
6,5,1939,5_foreign___,"[foreign, , , , , , , , , ]","[foreign, foreign, foreign]"
7,6,1655,6_weather_instrument_radar_conditions,"[weather, instrument, radar, conditions, cloud...",[the pilot departed on a visual flight rules v...
8,7,1326,7_knots_wind_winds_degrees,"[knots, wind, winds, degrees, gusting, runway,...",[the pilot of the tailwheel equipped airplane ...
9,8,1171,8_oil_connecting_rod_crankshaft,"[oil, connecting, rod, crankshaft, engine, bea...",[the pilot receiving instruction reported that...


In [24]:
topic_info.shape, len(df)

((279, 5), 87951)

In [25]:
# Create topic mapping
topic_lookup = {}
for idx, row in topic_info.iterrows():
    topic, name = row['Topic'], row['Name']
    topic_lookup[topic] = name
topic_lookup

{-1: '-1_airplane_runway_pilot_and',
 0: '0_acft_plt_rwy_flt',
 1: '1_helicopter_rotor_tail_collective',
 2: '2_student_instructor_solo_cfi',
 3: '3_factual_same_information_as',
 4: '4_fuel_tank_gallons_tanks',
 5: '5_foreign___',
 6: '6_weather_instrument_radar_conditions',
 7: '7_knots_wind_winds_degrees',
 8: '8_oil_connecting_rod_crankshaft',
 9: '9_unk___',
 10: '10_glider_tow_plane_spoilers',
 11: '11_water_lake_float_floats',
 12: '12_snow_covered_ski_ice',
 13: '13_fire_smoke_exhaust_electrical',
 14: '14_lines_wires_wire_field',
 15: '15_carburetor_icing_heat_dew',
 16: '16_weight_gross_pounds_takeoff',
 17: '17_balloon_basket_envelope_passengers',
 18: '18_brake_brakes_tire_right',
 19: '19_tailwheel_equipped_preaccident_rudder',
 20: '20_bounced_nose_hard_firewall',
 21: '21_power_engine_loss_reason',
 22: '22_gear_main_strut_fracture',
 23: '23_ice_icing_conditions_rime',
 24: '24_magneto_magnetos_engine_spark',
 25: '25_water_fuel_contamination_gascolator',
 26: '26_proba

In [26]:
df['topic_name'] = df['topic'].apply(lambda x: topic_lookup[x])

**Observation:** `BERTopic` created `285` unique categories using `87,951` reports. The top three categories are:
1) Airplanes, runways, and pilots at `24,304` reports
2) ACFT (Aircraft), PLT (Pilot), RWY (Runway), and FLT (Flight) at `14,311` reports
3) Helicopter, Rotor, Tail, and Collective at `6,362` reports.

Expanding acronyms would improve the richness of the topics as we can tell from the second category being composed of all acronyms. It's interesting that the 3rd category appears to have separated out all rotocraft related reports which is pretty nice.

### LDA with Visualization (Gensim)

In [27]:
# lda_results = group_topics_lda(
#     data=df['analysis_clean']
# )

## Export Dataset
Exporting for ingest into visualization applications such as Power BI.

In [28]:
df.columns

Index(['event_id', 'event_date', 'investigation_type', 'country',
       'aircraft_damage', 'aircraft_category', 'make', 'model',
       'amateur_built', 'number_of_engines', 'engine_type', 'far_description',
       'schedule', 'purpose_of_flight', 'total_fatal_injuries',
       'total_serious_injuries', 'total_minor_injuries', 'total_uninjured',
       'weather_condition', 'broad_phase_of_flight', 'analysis',
       'analysis_clean', 'analysis_preprocessed', 'city', 'longitude',
       'latitude', 'address', 'geometry', 'place', 'number_of_seats',
       'type_aircraft', 'type_engine', 'total_person',
       'far_description_factorized', 'schedule_factorized',
       'purpose_of_flight_factorized', 'make_factorized', 'model_factorized',
       'year', 'publication_year', 'month', 'publication_month', 'day',
       'publication_day', 'date_difference', 'publication_month_name',
       'event_month_name', 'season', 'sentiment_polarity',
       'sentiment_category', 'topic', 'topic_name'

In [29]:
df.to_csv("../data/enriched_ntsb_nlp.csv", index=False)

In [30]:
topic_info.to_csv('../data/ntsb_topic_info.csv', index=False)

In [31]:
# Create a zip archive containing the enriched dataset and topic info
import zipfile
import os

output_zip = '../data/ntsb_nlp_results.zip'

with zipfile.ZipFile(output_zip, 'w') as zf:
    zf.write('../data/enriched_ntsb_nlp.csv', os.path.basename('../data/enriched_ntsb_nlp.csv'))
    zf.write('../data/ntsb_topic_info.csv', os.path.basename('../data/ntsb_topic_info.csv'))

print(f'Created zip archive: {output_zip}')

Created zip archive: ../data/ntsb_nlp_results.zip
